# ShopSense 08: Check that every citation number points to the right review

**Goal:** in notebook 07 the model wrote mostly true sentences, but often put the wrong number on them (for example "lasts all day [4]" when the words are in review [5]). A review of 10 answers found 8 correct under a loose rule and only about 2 under a strict one. This notebook tests a **code-side fix** on the answers we already have. **No Ollama is needed.**

**What the fix does** (`src/shopsense/llm/attribute.py`)
1. **Quoted words.** If a clause quotes a phrase, the cited review must contain it. If not, the number moves to the review that does. A quote found in **no** review removes the sentence (a made-up quote).
2. **Meaning (off by default).** Each clause is compared with each excerpt using the MiniLM embeddings and the number is replaced by the best match. **First run result: this part did more harm than good** (it moved correct numbers to wrong reviews, because embeddings treat "easy to use" and "very hard to use" as similar), so `use_meaning=False` is now the default and only the quote check runs. To repeat the experiment set `USE_MEANING = True` in the run cell.
3. **Honest limits.** A sentence with a number inside its wording ("reviewers like [4] and [5] say...") is left alone and counted as `not_checked`. A clause that fits nothing is kept but counted as `weak`. This code cannot catch a made-up *idea* (for example "implying the packaging is poor"); only reading can.

**Reads:** `reports/eval/07_answers_vector.csv`, `reports/eval/07_hand_read.csv`.
**Writes:** `reports/eval/08_attribution_check.csv`.

### Rule written before running
We turn the fix on (`attribute=True` in `answer.py`) only if, among the answers it **changes**, you judge more **better** than **worse**. Until then it stays off.

## 1. Setup

**What this does:** imports libraries, loads the same MiniLM model the search uses (about 10 seconds), and loads the saved answers from notebook 07.

In [ ]:
import re
import sys
from pathlib import Path

import pandas as pd
from sentence_transformers import SentenceTransformer

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from shopsense.llm.attribute import reattribute

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 160)

OUT = ROOT / "reports" / "eval"
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

answers = pd.read_csv(OUT / "07_answers_vector.csv")
answers = answers[~answers["refused"] & (answers["answer"] != "I could not write a reliable answer from these reviews.")]
print("answered questions to check:", len(answers))

## 2. Run the check

**What this does:** applies the check to every saved answer. `passages` holds the six excerpts in `[1]` to `[6]` order. The result is kept next to the old answer.

In [ ]:
USE_MEANING = False      # True repeats the embedding experiment that did harm

rows = []
for row in answers.itertuples():
    excerpts = str(row.passages).split(" || ")
    new_answer, report = reattribute(row.answer, excerpts, embedder, use_meaning=USE_MEANING)
    rows.append({"id": row.id, "old_answer": row.answer, "new_answer": new_answer, "excerpts": excerpts, **report})

checked = pd.DataFrame(rows)
checked["changed"] = checked["old_answer"] != checked["new_answer"]
print(checked[["sentences", "not_checked", "citations_moved", "sentences_removed", "fabricated_quotes", "weak_clauses"]].sum().to_string())
print("answers changed:", int(checked["changed"].sum()), "of", len(checked))

**What this does:** prints every answer that changed, old and new, so you can see exactly what the code did.

In [ ]:
for row in checked[checked["changed"]].itertuples():
    print("\n", row.id)
    print("  OLD:", row.old_answer)
    print("  NEW:", row.new_answer)

## 3. Cases found by hand

**What this does:** two claims I found wrong when I read your 10 answers. We check that the fix repairs them. These cases are why the fix was written, so A third error (G018, "[4] and [5] find it medicinal") is not in the list because its numbers sit inside the wording, so the fix leaves it alone (the limit stated above). Passing these two is **not** evidence that it works on new answers; the section after this is the real test.

In [ ]:
known = [("G052", "all day", 5, 4), ("G036", "worked great", 5, 4)]

for question_id, snippet, wanted, unwanted in known:
    match = checked[checked["id"] == question_id]
    if match.empty:
        print(question_id, "not in this run")
        continue
    text = match.iloc[0]["new_answer"]
    start = text.find(snippet)
    group = re.search(r"(?:\[\d+\])+", text[start:]) if start >= 0 else None
    numbers = sorted(set(int(n) for n in re.findall(r"\d+", group.group()))) if group else []
    status = "PASS" if wanted in numbers and unwanted not in numbers else "FAIL"
    print(f"{question_id} '{snippet}': next citation is {numbers}, wanted {wanted}, not {unwanted} -> {status}")

## 4. The real test: you judge the changes

**What this does:** writes only the answers the fix changed to `08_attribution_check.csv`, with old and new answer and the excerpts. In `verdict` write one word:
- `better`: the new numbers point to the right reviews more than the old ones
- `same`: no real difference
- `worse`: a correct number was changed to a wrong one, or a good sentence was removed

Then run the last cell.

In [ ]:
review = checked[checked["changed"]].copy()
review["excerpts"] = review["excerpts"].apply(lambda parts: "\n".join(f"[{i}] {p}" for i, p in enumerate(parts, start=1)))
review["verdict"] = ""
review["note"] = ""
review[["id", "old_answer", "new_answer", "excerpts", "verdict", "note"]].to_csv(OUT / "08_attribution_check.csv", index=False)
print("written:", OUT / "08_attribution_check.csv", "| rows to judge:", len(review))

**Run this last cell only after you filled `verdict`.**

In [ ]:
done = pd.read_csv(OUT / "08_attribution_check.csv")
print(done["verdict"].value_counts(dropna=False).to_string())